# 10 Night-Targeted Delta Optimizer

07-09 showed that the strongest current rule is an incumbent-rebased soft mix:
keep the best r2 checkpoint and add only carefully selected later-round deltas.
The remaining weak point is night-domain quality.  This notebook searches
group-wise coefficients directly against night mini-slices, then validates the
best candidates on the full total and six day/night domain slices.


In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path('/app/Object_Detection')
PROJECT = ROOT / 'dynamic_quality_aware_classwise_aggregation' / 'moe_dqa_judger'
OUT = PROJECT / 'output' / '10_night_targeted_delta_optimizer'
OUT


PosixPath('/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/moe_dqa_judger/output/10_night_targeted_delta_optimizer')

In [2]:
import subprocess, sys

cmd = [
    sys.executable,
    str(PROJECT / 'scripts' / 'run_10_night_targeted_delta_optimizer.py'),
    '--workspace-root', str(OUT),
    '--rounds', '4,15,19,21',
    '--night-mini-images', '512',
    '--random-candidates', '1',
    '--template-topk', '1',
    '--full-eval-topk', '5',
    '--per-domain-topk', '1',
    '--max-full-candidates', '7',
    '--resume',
    '--notify-discord',
]
print(' '.join(cmd))
subprocess.run(cmd, cwd=ROOT, check=True)


/opt/venv/bin/python3 /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/moe_dqa_judger/scripts/run_10_night_targeted_delta_optimizer.py --workspace-root /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/moe_dqa_judger/output/10_night_targeted_delta_optimizer --rounds 4,15,19,21 --night-mini-images 512 --random-candidates 1 --template-topk 1 --full-eval-topk 5 --per-domain-topk 1 --max-full-candidates 7 --resume --notify-discord


{
  "manifest": {
    "created_utc": "2026-05-13T16:40:56.792459+00:00",
    "protocol": "dqa_softmox_night_targeted_delta_optimizer_v1",
    "formula": "I_best + alpha*(A_t-G_t) + beta*(S_t-G_t)",
    "rounds": [
      4,
      15,
      19,
      21
    ],
    "night_mini_images": 512,
    "objective": "night mini mean + worst + mAP50 + recall, then full total/domain validation"
  },
  "top_night": [
    {
      "label": "r004_scaled04_best01_rand000_075",
      "round": 4,
      "candidate_id": "scaled04_best01_rand000_075",
      "phase": "prior",
      "night_probe_mean_score": 0.46035000000000004,
      "night_probe_worst_score": 0.38680000000000003,
      "night_probe_mean_map50": 0.373,
      "night_probe_mean_recall": 0.3353333333333333,
      "body_a": -0.1875,
      "body_s": -0.1875,
      "head_a": -0.1875,
      "head_s": 0.2084823002319294,
      "router_a": 0.34636626091159817,
      "router_s": -0.12146159455565633,
      "expert0_a": 0.45021717986441123,
      "expert

CompletedProcess(args=['/opt/venv/bin/python3', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/moe_dqa_judger/scripts/run_10_night_targeted_delta_optimizer.py', '--workspace-root', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/moe_dqa_judger/output/10_night_targeted_delta_optimizer', '--rounds', '4,15,19,21', '--night-mini-images', '512', '--random-candidates', '1', '--template-topk', '1', '--full-eval-topk', '5', '--per-domain-topk', '1', '--max-full-candidates', '7', '--resume', '--notify-discord'], returncode=0)

In [3]:
night = pd.read_csv(OUT / 'stats' / '10_night_probe_summary.csv')
display(night.head(12)[['label','night_objective','night_probe_mean_score','night_probe_worst_score','night_probe_mean_map50']])

summary = pd.read_csv(OUT / 'stats' / '10_full_domain_summary.csv')
display(summary[['label','total_score','day_mean_score','night_mean_score','worst_domain_score','group_dro_score','night_mean_map50']])

router = pd.read_csv(OUT / 'stats' / '10_domain_router_summary.csv')
display(router)

print((OUT / '10_night_targeted_delta_optimizer_report.md').read_text())


,label,night_objective,night_probe_mean_score,night_probe_worst_score,night_probe_mean_map50
0,r004_scaled04_best01_rand000_075,0.415254,0.460350,0.38680,0.373000
1,r021_city_night_recall,0.414839,0.459850,0.38640,0.372333
2,r004_head_repair_moe_target,0.414814,0.459633,0.38675,0.372000
3,r004_anti_drift_moe_only,0.414814,0.459633,0.38675,0.372000
4,r015_rand000,0.414787,0.459733,0.38640,0.372333
5,r015_tiny_repair_delta,0.414755,0.459617,0.38665,0.372000
6,r004_city_night_recall,0.414721,0.459467,0.38665,0.372333
7,r019_residential_night_precision,0.414706,0.459517,0.38640,0.372333
8,r019_anti_drift_moe_only,0.414682,0.459500,0.38640,0.372333
9,r004_highway_night_guard,0.414664,0.459733,0.38605,0.372333


,label,total_score,day_mean_score,night_mean_score,worst_domain_score,group_dro_score,night_mean_map50
0,r021_city_night_recall,0.57455,0.616650,0.442417,0.38425,0.465630,0.358000
1,r004_scaled04_best01_rand000_075,0.57455,0.617433,0.442183,0.38400,0.465597,0.358000
2,r004_head_repair_moe_target,0.57450,0.616700,0.442200,0.38460,0.465580,0.357667
3,r004_anti_drift_moe_only,0.57455,0.616617,0.442200,0.38460,0.465563,0.357667
4,r015_rand000,0.57455,0.616717,0.442083,0.38425,0.465443,0.357667
5,incumbent_r002,0.57455,0.616700,0.442083,0.38425,0.465440,0.357667
6,r015_residential_night_precision,0.57355,0.616100,0.442167,0.38425,0.465370,0.358000


,day_mean_score,domain_mean_map50,domain_mean_score,group_dro_score,night_mean_map50,night_mean_score,policy,worst_domain_score
0,0.61670,0.426333,0.529392,0.46544,0.357667,0.442083,incumbent_r002,0.38425
1,0.61745,0.427000,0.530042,0.46599,0.358333,0.442633,night_domain_router,0.38460


# DQA-SoftMoX Night-Targeted Delta Optimizer 10

- created_utc: 2026-05-13T17:09:06.601183+00:00
- rounds: 4,15,19,21
- search: incumbent-rebased coefficients on night mini-slices
- baseline total score: 0.57455

## Top Night Probe Candidates

| rank | candidate | objective | night mean | night worst | night mAP50 |
|---:|---|---:|---:|---:|---:|
| 1 | r004_scaled04_best01_rand000_075 | 0.41525 | 0.46035 | 0.38680 | 0.373 |
| 2 | r021_city_night_recall | 0.41484 | 0.45985 | 0.38640 | 0.372 |
| 3 | r004_head_repair_moe_target | 0.41481 | 0.45963 | 0.38675 | 0.372 |
| 4 | r004_anti_drift_moe_only | 0.41481 | 0.45963 | 0.38675 | 0.372 |
| 5 | r015_rand000 | 0.41479 | 0.45973 | 0.38640 | 0.372 |
| 6 | r015_tiny_repair_delta | 0.41475 | 0.45962 | 0.38665 | 0.372 |
| 7 | r004_city_night_recall | 0.41472 | 0.45947 | 0.38665 | 0.372 |
| 8 | r019_residential_night_precision | 0.41471 | 0.45952 | 0.38640 | 0.372 |
| 9 | r019_anti_drift_moe_only | 0.41468 | 0.45950 | 0.38640 | 0.372 |
| 10 | r004